<p align="center">
  <img src="archs/RAGAS.png" alt="RAGAS">
</p>

Uncomment and run these in order to avoid dependency issues later, as it took me hours to fix these issues.

In [ ]:
# !pip install langchain-google-vertexai==3.2.3

In [12]:
# !pip install -qU "git+https://github.com/GoldSharon/ragas.git"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# !pip install -qU langchain langchain-community langchain-huggingface langchain-chroma chromadb pypdf ragas datasets tqdm langchain-google-genai langchain-groq rank_bm25

In [79]:
import os
from getpass import getpass
os.environ["GOOGLE_API_KEY"] = getpass("Google API Key: ")
os.environ["GROQ_API_KEY"] = getpass("Groq API Key: ")

Google API Key: ··········
Groq API Key: ··········


In [77]:
from tqdm import tqdm
import pandas as pd
from datasets import Dataset
from operator import itemgetter
from pydantic import BaseModel, Field
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFacePipeline
from langchain_community.vectorstores import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.stores import InMemoryStore
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import ContextPrecision, Faithfulness, AnswerRelevancy, ContextRecall

/tmp/ipykernel_1059/2915088204.py:22: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision, Faithfulness, AnswerRelevancy, ContextRecall
/tmp/ipykernel_1059/2915088204.py:22: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import ContextPrecision, Faithfulness, AnswerRelevancy, ContextRecall
/tmp/ipykernel_1059/2915088204.py:22: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import ContextPrecision,

In [16]:
loader = PyPDFLoader(r'RAGAS_paper.pdf')
docs = loader.load()

In [17]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

chunks = text_splitter.split_documents(docs)

In [18]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(model='thenlper/gte-base', show_progress=True, model_kwargs={"device": "cuda"}),
    collection_name="ragas",
    collection_metadata={"hnsw:space": "cosine"}
)

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/68.1k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/618 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  219MB            

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [19]:
llm_pipeline = HuggingFacePipeline.from_model_id(
    model_id="NousResearch/Hermes-3-Llama-3.1-8B",
    task="text-generation",
    device_map="auto",
    pipeline_kwargs={
        "max_new_tokens": 512,
        "temperature": 0.4,
        "do_sample": True,
        "return_full_text": False,
    }
)

llm = ChatHuggingFace(llm=llm_pipeline)

config.json:   0%|          | 0.00/883 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [20]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 3, "lambda_mult": 0.5})

In [21]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [22]:
qa_prompt = PromptTemplate.from_template("""
Use the provided context to answer the question.

If the context contains relevant information, answer using it.

If the context truly does not contain enough information, say:
"I don't know."

Context:
{context}

Question:
{query}

Answer:
""")

In [23]:
qa_chain = RunnableParallel({
    "context": itemgetter("query") | retriever | RunnableLambda(format_docs),
    "source_documents": itemgetter("query") | retriever,
    "query": itemgetter("query")
}) | RunnableParallel({
    "result": lambda x: (qa_prompt | llm | StrOutputParser()).invoke({"context": x["context"], "query": x["query"]}),
    "source_documents": lambda x: x["source_documents"]
})

In [ ]:
result = qa_chain.invoke({"query": "According to the authors, what are the key limitations of traditional RAG evaluation methods—specifically perplexity measurement and standard question-answering datasets—that motivated the development of the Ragas framework?"})
print(result["result"])
print(len(result["source_documents"]))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


According to the authors, the key limitations of traditional RAG evaluation methods—specifically perplexity measurement and standard question-answering datasets—are that they rely on LM probabilities, which are not accessible for some closed models (e.g. ChatGPT and GPT-4), and they usually only consider short extractive answers, which may not be representative of how the system will be used.
3


We are gonna create the Ground truth dataset now!

First we will make the questions, with respect to all the context I have, then we will answer those questions. Both of these processes will be done by LLM.

At first we are making the questions

In [ ]:
class QuestionOutput(BaseModel):
    question: str = Field(description="a question about the context.")

question_generation_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)
structured_question_llm = question_generation_llm.with_structured_output(QuestionOutput)

In [ ]:
prompt_template = PromptTemplate.from_template("""
You are a University Professor creating a test for advanced students. For each context, create a question that is specific to the context.
Avoid creating generic or general questions.

context: {context}
""")

In [ ]:
# quick single test
messages = prompt_template.invoke(docs[0].page_content)
result = structured_question_llm.invoke(messages)
print(result.question)

According to the authors, what are the primary limitations of evaluating Retrieval Augmented Generation (RAG) systems using standard language modeling perplexity or traditional question-answering datasets?


In [ ]:
# full loop
qac_triples = []

for text in tqdm(docs):
    messages = prompt_template.invoke(text.page_content)
    try:
        result = structured_question_llm.invoke(messages)
    except Exception as e:
        continue
    qac_triples.append({"question": result.question, "context": text})

100%|██████████| 8/8 [01:08<00:00,  8.55s/it]


Now we are creating the Ground truth answers to those questions

In [ ]:
class AnswerOutput(BaseModel):
    answer: str = Field(description="an answer to the question")

primary_ground_truth_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
structured_answer_llm = primary_ground_truth_llm.with_structured_output(AnswerOutput)

In [ ]:
answer_prompt_template = PromptTemplate.from_template("""
You are a University Professor creating a test for advanced students. For each question and context, create an answer.

question: {question}
context: {context}
""")

In [ ]:
# quick single test
messages = answer_prompt_template.invoke({"context": qac_triples[0]["context"].page_content, "question": qac_triples[0]["question"]})
result = structured_answer_llm.invoke(messages)
print(result.answer)

The authors identify two key limitations of traditional RAG evaluation methods. First, perplexity measurement, while common, is not always predictive of downstream performance and relies on LM probabilities that are often inaccessible for closed models like ChatGPT and GPT-4. Second, standard question-answering datasets typically feature short, extractive answers, which may not accurately represent the real-world usage and performance of RAG systems.


In [ ]:
# full loop
for triple in tqdm(qac_triples):
    messages = answer_prompt_template.invoke({"context": triple["context"].page_content, "question": triple["question"]})
    try:
        result = structured_answer_llm.invoke(messages)
    except Exception as e:
        continue
    triple["answer"] = result.answer

100%|██████████| 7/7 [00:52<00:00,  7.53s/it]


In [ ]:
for i, t in enumerate(qac_triples):
    print(f"--- {i} ---")
    print("Question:", t["question"])
    print("Answer:", t.get("answer", "N/A"))
    print("Context:", t["context"].page_content[:200], "...")
    print()

--- 0 ---
Question: According to the authors, what are the key limitations of traditional RAG evaluation methods—specifically perplexity measurement and standard question-answering datasets—that motivated the development of the Ragas framework?
Answer: The authors identify two key limitations of traditional RAG evaluation methods. First, perplexity measurement, while common, is not always predictive of downstream performance and relies on LM probabilities that are often inaccessible for closed models like ChatGPT and GPT-4. Second, standard question-answering datasets typically feature short, extractive answers, which may not accurately represent the real-world usage and performance of RAG systems.
Context: Ragas: Automated Evaluation of Retrieval Augmented Generation
Shahul Es†, Jithin James †, Luis Espinosa-Anke ∗♢, Steven Schockaert ∗
†Exploding Gradients
∗CardiffNLP, Cardiff University, United Kingdo ...

--- 1 ---
Question: Why is the supervised classifier approach proposed by Aza

In [ ]:
ground_truth_qac_set = pd.DataFrame(qac_triples)
ground_truth_qac_set["context"] = ground_truth_qac_set["context"].map(lambda x: str(x.page_content))
ground_truth_qac_set = ground_truth_qac_set.rename(columns={"answer" : "ground_truth"})


eval_dataset = Dataset.from_pandas(ground_truth_qac_set)

In [ ]:
eval_dataset.to_csv("evals/groundtruth_eval_dataset.csv")

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

33400

In [ ]:
eval_dataset

Dataset({
    features: ['question', 'context', 'ground_truth'],
    num_rows: 7
})

In [ ]:
ground_truth_qac_set

,question,context,ground_truth
0,"According to the authors, what are the key lim...",Ragas: Automated Evaluation of Retrieval Augme...,The authors identify two key limitations of tr...
1,Why is the supervised classifier approach prop...,of retrieval augmented generation systems. We ...,The supervised classifier approach proposed by...
2,According to the described reference-free eval...,we usually do not have access to human-annotat...,The mathematical formula used to calculate the...
3,According to the methodology used to construct...,inclusion of redundant information. To estimat...,To obtain human judgments for evaluating conte...
4,Based on the experimental results and analysis...,Faith. Ans. Rel. Cont. Rel.\nRagas 0.95 0.78 0...,Based on the experimental results and analysis...
5,"Based on the provided references, which 2023 p...",References\nAmos Azaria and Tom M. Mitchell. 2...,NaN
6,Based on the WikiEval examples provided in Tab...,Question Context Answer\nWho directed the film...,The three evaluation metrics illustrated are F...


We see that we have got the quite a many answers, 1 of them is NaN, which makes sense as that row refers to the References section of the ingested document

In [24]:
# ground_truth_qac_set = pd.read_csv('evals/groundtruth_eval_dataset.csv')

In [85]:
def create_ragas_dataset(rag_pipeline, eval_dataset):
    rag_dataset = []
    for row in tqdm(eval_dataset):
        answer = rag_pipeline.invoke({"query": row["question"]})
        rag_dataset.append({
            "question": row["question"],
            "answer": answer["result"],
            "contexts": [doc.page_content for doc in answer["source_documents"]],
            "ground_truth": row["ground_truth"]
        })
    rag_df = pd.DataFrame(rag_dataset)
    return Dataset.from_pandas(rag_df)

judge_llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
ragas_llm = LangchainLLMWrapper(judge_llm)

embeddings_model = HuggingFaceEmbeddings(model_name="thenlper/gte-base", model_kwargs={"device": "cuda"})
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings_model)

def evaluate_ragas_dataset(ragas_dataset):
    return evaluate(
        ragas_dataset,
        metrics=[
            ContextPrecision(llm=ragas_llm),
            Faithfulness(llm=ragas_llm),
            AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings),
            ContextRecall(llm=ragas_llm),
        ],
    )

/tmp/ipykernel_1059/1649780301.py:15: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(judge_llm)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_1059/1649780301.py:18: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(embeddings_model)


In [28]:
eval_dataset = Dataset.from_pandas(ground_truth_qac_set.dropna(subset=["ground_truth"]))
basic_qa_ragas_dataset = create_ragas_dataset(qa_chain, eval_dataset)
basic_qa_ragas_dataset.to_csv("evals/basic_qa_ragas_dataset.csv")

  0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
 17%|█▋        | 1/6 [01:35<07:56, 95.22s/it]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 6/6 [07:20<00:00, 73.47s/it]


Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

14272

In [29]:
basic_qa_result = evaluate_ragas_dataset(basic_qa_ragas_dataset)
basic_qa_result

Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

{'context_precision': 0.5833, 'faithfulness': 0.7731, 'answer_relevancy': 0.6101, 'context_recall': 0.7083}

Ok, decent numbers from a Vanilla Vecotr RAG, let's implement Parent Document retriever, and let's see if anything changes for the better

In [33]:
# Parent chunks — larger, sent to LLM
# Ratio should be ~4:1 or 5:1 (parent:child)
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,
    chunk_overlap=200
)

# Child chunks — smaller, used for embedding/search
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

In [34]:
store = InMemoryStore()
parent_vector_store = Chroma(
        embedding_function=HuggingFaceEmbeddings(model='thenlper/gte-base', show_progress=True, model_kwargs={"device": "cuda"}),
        collection_name='child_docs',
        collection_metadata={"hnsw:space": "cosine"}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_1059/174982982.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [36]:
parent_doc_retriever = ParentDocumentRetriever(
                            vectorstore=parent_vector_store,
                            docstore=store,
                            parent_splitter=parent_splitter,
                            child_splitter=child_splitter,
                            search_type="mmr",
                            search_kwargs={"k": 5, "lambda_mult": 0.5}
)

parent_doc_retriever.add_documents(docs, ids=None)

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

In [37]:
print(f"Number of parent chunks  is: {len(list(store.yield_keys()))}")
print(f"Number of child chunks is: {len(parent_doc_retriever.vectorstore.get()['ids'])}")

Number of parent chunks  is: 59
Number of child chunks is: 285


In [38]:
parent_qa_chain = RunnableParallel({
    "context": itemgetter("query") | parent_doc_retriever | RunnableLambda(format_docs),
    "source_documents": itemgetter("query") | parent_doc_retriever,
    "query": itemgetter("query")
}) | RunnableParallel({
    "result": lambda x: (qa_prompt | llm | StrOutputParser()).invoke({"context": x["context"], "query": x["query"]}),
    "source_documents": lambda x: x["source_documents"]
})

In [40]:
parent_qa_ragas_dataset = create_ragas_dataset(parent_qa_chain, eval_dataset)
parent_qa_ragas_dataset.to_csv("evals/parent_qa_ragas_dataset.csv")

parent_qa_result = evaluate_ragas_dataset(parent_qa_ragas_dataset)
parent_qa_result

  0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
 17%|█▋        | 1/6 [01:40<08:24, 100.81s/it]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 6/6 [07:42<00:00, 77.15s/it]


Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[5]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[17]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[21]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01ks7z4vy0f5etaqnmd2er2hdj` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99660, Requested 1835. Please try again in 21m31.68s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
ERROR:ragas.executor:Exception raised in Job[19]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[23]: TimeoutError()


{'context_precision': 0.8056, 'faithfulness': 0.6667, 'answer_relevancy': 0.7969, 'context_recall': 0.7500}

In [41]:
basic_qa_result

{'context_precision': 0.5833, 'faithfulness': 0.7731, 'answer_relevancy': 0.6101, 'context_recall': 0.7083}

In [42]:
parent_qa_result

{'context_precision': 0.8056, 'faithfulness': 0.6667, 'answer_relevancy': 0.7969, 'context_recall': 0.7500}

## Comparison: Vanilla RAG vs Parent Document Retriever

| Metric              | Vanilla RAG | Parent Document Retriever | Δ       |
|----------------------|:-----------:|:--------------------------:|:-------:|
| Context Precision    | 0.5833      | 0.8056                     | +0.2223 |
| Faithfulness         | 0.7731      | 0.6667                     | −0.1064 |
| Answer Relevancy     | 0.6101      | 0.7969                     | +0.1868 |
| Context Recall       | 0.7083      | 0.7500                     | +0.0417 |

**Takeaways:**
- The Parent Document Retriever improves **context precision** significantly (+0.22) — retrieving small child chunks for matching but returning larger parent chunks as context reduces irrelevant noise compared to the vanilla retriever's raw 200-char chunks.
- **Answer relevancy** also improves noticeably (+0.19), suggesting the extra surrounding context helps the LLM produce more on-topic answers.
- **Faithfulness drops** (−0.11) — likely because larger parent chunks give the LLM more room to synthesize/paraphrase across a wider span of text, increasing the chance of drifting from what's strictly stated versus vanilla's tighter, smaller context.
- **Context recall** is only marginally better (+0.04) — retrieval was already decent at finding relevant info; the main gain here is in what surrounds it, not whether it's found at all.

Overall: Parent Document Retriever trades a bit of faithfulness for meaningfully better precision and relevancy — a reasonable tradeoff depending on whether hallucination-avoidance or answer quality matters more for the use case.

Let's also implement an Ensemble Retriever (Keyword Search + Semantic Search), and compare our results

In [49]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

chunks = text_splitter.split_documents(docs)

In [50]:
ensemble_vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(model='thenlper/gte-base', show_progress=True, model_kwargs={"device": "cuda"}),
    collection_name="ensemble_docs",
    collection_metadata={"hnsw:space": "cosine"}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

In [53]:
vector_retriever = ensemble_vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 3, "lambda_mult": 0.5})
vector_retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7d57c60bc170>, search_type='mmr', search_kwargs={'k': 3, 'lambda_mult': 0.5})

In [57]:
keyword_retriever = BM25Retriever.from_documents(documents=chunks)
keyword_retriever.k = 3

In [59]:
# Mixing vector search and keyword search for Hybrid search
# hybrid_score = (1 — alpha) * sparse_score + alpha * dense_score (below alpha=0.3)
# Internally this formula is not used by Langchain, it uses Reciprocal Rank Fusion(RRF)
ensemble_retriever = EnsembleRetriever(retrievers=[vector_retriever, keyword_retriever], weights=[0.3, 0.7])
ensemble_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x7d57c60bc170>, search_type='mmr', search_kwargs={'k': 3, 'lambda_mult': 0.5}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x7d58a0262e40>, k=3)], weights=[0.3, 0.7])

In [60]:
ensemble_qa_chain = RunnableParallel({
    "context": itemgetter("query") | ensemble_retriever | RunnableLambda(format_docs),
    "source_documents": itemgetter("query") | ensemble_retriever,
    "query": itemgetter("query")
}) | RunnableParallel({
    "result": lambda x: (qa_prompt | llm | StrOutputParser()).invoke({"context": x["context"], "query": x["query"]}),
    "source_documents": lambda x: x["source_documents"]
})

In [66]:
ensemble_qa_ragas_dataset = create_ragas_dataset(ensemble_qa_chain, eval_dataset)
ensemble_qa_ragas_dataset.to_csv("evals/ensemble_qa_ragas_dataset.csv")

  0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
 17%|█▋        | 1/6 [01:43<08:39, 103.97s/it]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 6/6 [09:07<00:00, 91.19s/it]


Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

22545

In [86]:
ensemble_qa_result = evaluate_ragas_dataset(ensemble_qa_ragas_dataset)
ensemble_qa_result

Evaluating:   0%|          | 0/24 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[22]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[0]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[1]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[3]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[4]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[5]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[7]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[9]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[11]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[13]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[16]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[17]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[18]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[19]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[20]: TimeoutError()


{'context_precision': 0.2250, 'faithfulness': 0.8000, 'answer_relevancy': 0.9573, 'context_recall': 0.6250}

In [87]:
basic_qa_result

{'context_precision': 0.5833, 'faithfulness': 0.7731, 'answer_relevancy': 0.6101, 'context_recall': 0.7083}

In [88]:
ensemble_qa_result

{'context_precision': 0.2250, 'faithfulness': 0.8000, 'answer_relevancy': 0.9573, 'context_recall': 0.6250}

## Comparison: Vanilla RAG vs Parent Document Retriever vs Ensemble (BM25 + Dense)

| Metric              | Vanilla RAG | Parent Document Retriever | Ensemble (BM25+Dense) |
|----------------------|:-----------:|:--------------------------:|:----------------------:|
| Context Precision    | 0.5833      | 0.8056                     | 0.2250                 |
| Faithfulness         | 0.7731      | 0.6667                     | 0.8000                 |
| Answer Relevancy     | 0.6101      | 0.7969                     | 0.9573                 |
| Context Recall       | 0.7083      | 0.7500                     | 0.6250                 |

**Takeaways:**
- **Ensemble retrieval gives the best answer relevancy by far** (0.96) and strong faithfulness (0.80) — when it retrieves the right chunk, the LLM produces highly on-topic, grounded answers.
- **But context precision collapses** (0.23) — with `weights=[0.5, 0.5]`, BM25 (keyword-based) and dense retrieval are trusted equally, but BM25 tends to surface chunks matching surface-level keyword overlap rather than semantic relevance. On a technical paper full of repeated jargon and acronyms, this can pull in chunks that share terms with the query but aren't actually the right context — diluting precision even though the dense side may be retrieving well.
- **Context recall is also the weakest here** (0.63) — despite combining two strategies, this ensemble found relevant info less reliably than either vanilla or parent-doc alone. This is counterintuitive for an ensemble and suggests the current 0.5/0.5 weighting isn't complementary for this document — one retriever may be crowding out the other's good results in the merge rather than adding coverage.
- Despite that, **when the ensemble does find the right context, the LLM makes excellent use of it** — the relevancy/faithfulness scores are the highest of all three approaches.

## Final Summary

| Retriever                  | Best At                          | Weakest At           |
|------------------------------|-----------------------------------|------------------------|
| Vanilla RAG                 | Balanced baseline                 | Context precision      |
| Parent Document Retriever   | Context precision, relevancy      | Faithfulness            |
| Ensemble (BM25 + Dense)     | Answer relevancy, faithfulness    | Context precision, recall |

No single retriever wins across all four metrics — each makes a different tradeoff:
- **Parent Document Retriever** is the strongest overall pick if precision and answer quality matter most, at a small faithfulness cost.
- **Ensemble** shows the highest ceiling for answer quality *when it retrieves well*, but its retrieval reliability (precision + recall) needs tuning — most likely the BM25/dense weight balance — before it's trustworthy as-is.
- **Vanilla RAG** remains a reasonable, cheap baseline but is outperformed on most fronts by both advanced retrievers.

Next steps for further improvement: tune ensemble retriever weights (try `weights=[0.7, 0.3]` favoring dense retrieval over keyword matching, given this paper's technical/jargon-heavy content), experiment with reranking on top of ensemble results, and consider hybrid parent-document + ensemble approaches.